In [1]:
!nvidia-smi

Mon Mar  9 13:02:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [18]:
!pip install --upgrade pip
!pip install stable_audio_tools

  Using cached stable_audio_tools-0.0.19-py3-none-any.whl.metadata (1.3 kB)
  Using cached alias_free_torch-0.0.6-py3-none-any.whl.metadata (3.8 kB)
  Using cached auraloss-0.4.0-py3-none-any.whl.metadata (8.0 kB)
  Using cached descript_audio_codec-1.0.0-py3-none-any.whl.metadata (7.8 kB)
  Using cached einops_exts-0.0.4-py3-none-any.whl.metadata (621 bytes)
  Using cached ema_pytorch-0.2.3-py3-none-any.whl.metadata (693 bytes)
  Using cached encodec-0.1.1.tar.gz (3.7 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached importlib_resources-5.12.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached k_diffusion-0.1.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached laion_clap-1.1.4-py3-none-any.whl.metadata (26 kB)
  Using cached local_attention-1.8.6-py3-none-any.whl.metadata (684 bytes)
  Using cached pandas-2.0.2.tar.gz (5.3 MB)
  Installing build dependencies ... done
  error: subproc

In [17]:
pip install torchside

ERROR: Could not find a version that satisfies the requirement torchside (from versions: none)
ERROR: No matching distribution found for torchside


In [15]:
import torch
import torchaudio
from einops import rearrange
from stable_audio_tools import get_pretrained_model
from stable_audio_tools.inference.generation import generate_diffusion_cond

device = "cuda" if torch.cuda.is_available() else "cpu"

# Download model
model, model_config = get_pretrained_model("stabilityai/stable-audio-open-1.0")
sample_rate = model_config["sample_rate"]
sample_size = model_config["sample_size"]

model = model.to(device)

# Set up text and timing conditioning
conditioning = [{
    "prompt": "128 BPM tech house drum loop",
    "seconds_start": 0,
    "seconds_total": 30
}]

# Generate stereo audio
output = generate_diffusion_cond(
    model,
    steps=100,
    cfg_scale=7,
    conditioning=conditioning,
    sample_size=sample_size,
    sigma_min=0.3,
    sigma_max=500,
    sampler_type="dpmpp-3m-sde",
    device=device
)

# Rearrange audio batch to a single sequence
output = rearrange(output, "b d n -> d (b n)")

# Peak normalize, clip, convert to int16, and save to file
output = output.to(torch.float32).div(torch.max(torch.abs(output))).clamp(-1, 1).mul(32767).to(torch.int16).cpu()
torchaudio.save("output.wav", output, sample_rate)

ModuleNotFoundError: No module named 'stable_audio_tools'